# SentinelXAI — Train on CICIDS2017 + REAL-LIFE raw-data test (Google Colab)

This notebook runs the **actual SentinelXAI repo pipeline** (no duplicated logic):

```
raw CICIDS2017 CSVs
      |
      v
scripts/build_dataset.py          # merge + clean + stratified split
      |
      v
scripts/engineer_features.py      # feature engineering (fit-free transforms)
      |
      v
scripts/train_final_lightgbm.py   # tuned LightGBM (uses committed Optuna best params)
      |
      v
[REAL-LIFE TEST]  predict on RAW, uncleaned rows (NaN / Infinity / duplicates
                  that the pipeline would have filtered) — a true robustness check
      |
      v
download model artifacts  ->  unzip into your local repo (models/ + reports/)
```

> Why Colab? The full CICIDS2017 zip is ~50 GB and needs ~12 GB+ RAM to build.
> Colab's free tier (high-RAM CPU) handles it; no GPU required.


## 0 · Runtime prep

No GPU needed. Recommended runtime: **CPU**, and if available use
**Runtime → Change runtime type → High-RAM** (or a Pro/Pro+ high-RAM machine).
The heavy steps are single-threaded-ish, so a plain CPU runtime is fine.


In [1]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules
print("Running in Google Colab:", IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')   # optional — used only if you want a Drive copy


Mounted at /content/drive


## 1 · Get the SentinelXAI repo

In [2]:
# Repo clone — CHANGE the URL to your repository.
REPO_URL = "https://github.com/modak10/SentinelXAI.git"

if os.path.exists("/content/SentinelXAI/.git"):
    print("Repo already present — skipping clone.")
else:
    !git clone --depth 1 {REPO_URL} /content/SentinelXAI
%cd /content/SentinelXAI
!ls scripts/


Cloning into '/content/SentinelXAI'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 116 (delta 9), reused 96 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 804.94 KiB | 6.10 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/SentinelXAI
build_dataset.py       engineer_features.py  train_final_lightgbm.py
compare_all_models.py  run_eda.py	     tune_lightgbm.py
compare_baselines.py   train_baselines.py


In [3]:
# FALLBACK — if the clone failed (private repo / wrong URL), upload the repo as a zip instead.
# 1) On your machine:  Compress-Archive -Path * -Path zip (root must contain scripts/, src/, configs/)
# 2) Run this cell and pick the zip.
from google.colab import files
import zipfile, io

if not os.path.exists("/content/SentinelXAI/scripts/build_dataset.py"):
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as z:
        z.extractall("/content/SentinelXAI")
    os.chdir("/content/SentinelXAI")
    print("Extracted:", os.listdir("."))
else:
    print("Repo already present — skipping upload.")


Repo already present — skipping upload.


## 2 · Install dependencies

In [4]:
!pip install -q -r requirements.txt
# If a pinned version clashes with Colab's Python, relax it, e.g.:
#   !pip install -q lightgbm pandas numpy scikit-learn shap fastapi uvicorn pydantic python-multipart plotly joblib optuna
!python scripts/build_dataset.py --help >/dev/null 2>&1 && echo "pipeline scripts OK"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 116.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 94.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 123.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 107.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.9/153.9 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.8/301.8 kB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 96.3 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 11.2 MB/s eta

## 3 · Get the RAW CICIDS2017 data

Choose ONE method and run the matching cell. The files must land in
`data/raw/MachineLearningCVE/*.csv` (the 8 `*.pcap_ISCX.csv` files) because
`configs/config.yaml` points `data_raw_dir` there and `scripts/build_dataset.py`
merges everything in that folder.


### 3a · Recommended — Kaggle API (resumable, fastest)

In [ ]:
# 1) Upload kaggle.json (from https://www.kaggle.com/settings -> Account -> Create New API Token)
# 2) Run this cell.
from google.colab import files
import os, pathlib

if not os.path.exists("/content/kaggle.json"):
    uploaded = files.upload()
    os.rename(list(uploaded.keys())[0], "/content/kaggle.json")
os.makedirs("/root/.kaggle", exist_ok=True)
os.system("mv /content/kaggle.json /root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)

# Datasets that mirror the CICIDS2017 MachineLearningCVE CSVs:
KAGGLE_DATASET = "mrkmak/cicids-2017-dataset"   # <- change if you prefer another mirror
!mkdir -p data/raw/MachineLearningCVE
!kaggle datasets download -d {KAGGLE_DATASET} --unzip -p data/raw/MachineLearningCVE
!ls data/raw/MachineLearningCVE | head


### 3b — Direct link (edit the URL to a real, resumable mirror)

In [ ]:
import os
ZIP_URL = "PASTE_DIRECT_ZIP_URL_HERE"   # e.g. a university / mirror .zip link
RAW_DIR = "data/raw/MachineLearningCVE"
os.makedirs(RAW_DIR, exist_ok=True)
if ZIP_URL.startswith("http"):
    !wget -c -q --show-progress {ZIP_URL} -O MachineLearningCSV.zip
    !unzip -q -o MachineLearningCSV.zip -d {RAW_DIR}
    # if the zip contains a nested folder, flatten it:
    # !find data/raw/MachineLearningCVE -name "*.csv" -exec mv {} data/raw/MachineLearningCVE/ ;
else:
    print("Set ZIP_URL first.")
!ls {RAW_DIR} | head


### 3c — Manual upload (smallest, slowest)

In [ ]:
from google.colab import files
import io, zipfile

RAW_DIR = "data/raw/MachineLearningCVE"
import os; os.makedirs(RAW_DIR, exist_ok=True)
uploaded = files.upload()          # pick MachineLearningCSV.zip (or the CSVs)
for name, blob in uploaded.items():
    if name.endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(blob)) as z:
            z.extractall(RAW_DIR)
    elif name.endswith(".csv"):
        open(os.path.join(RAW_DIR, name), "wb").write(blob)
print(os.listdir(RAW_DIR)[:5])


## 4 · Build the processed dataset

Runs the real cleaning pipeline: merge the 8 CSVs, replace ±Infinity, drop
NaN/duplicate rows, normalize labels, stratified 70/15/15 split, and writes
`data/processed/{train,val,test}.parquet` + `data_quality_report.json`.
Expect **~5-15 minutes** and peak RAM ~12 GB on the full dataset.


In [ ]:
# Increase Colab's default matplotlib/temp limits aren't needed here; just run it.
!python scripts/build_dataset.py


## 5 · Feature engineering (fit-free transforms from configs/features.yaml)

In [ ]:
!python scripts/engineer_features.py


## 6 · Train the final LightGBM

Uses the **committed** Optuna best params in `reports/lightgbm/optuna_best_params.json`
(no re-tuning needed). Evaluates on VAL only — test stays untouched until final
selection (see docs/JUDGE_QNA.md Q8). Outputs:
`models/lightgbm/{lightgbm.joblib,label_encoder.joblib}` + `reports/lightgbm/*`.


In [ ]:
!python scripts/train_final_lightgbm.py


In [ ]:
# OPTIONAL — only if you want to re-tune hyperparameters (long, ~1-2h):
# !python scripts/tune_lightgbm.py
# Then re-run the training cell above so it picks up the new best params.


In [ ]:
import os
artifacts = [
    "models/lightgbm/lightgbm.joblib",
    "models/lightgbm/label_encoder.joblib",
    "reports/lightgbm/lightgbm_feature_list.json",
    "reports/lightgbm/lightgbm_metrics.json",
]
for p in artifacts:
    ok = os.path.exists(p)
    size_kb = os.path.getsize(p) // 1024 if ok else 0
    print(("OK  " if ok else "MISSING ") + f"{p}  {size_kb} KB")


## 7 · REAL-LIFE test on RAW (uncleaned) CICIDS2017 rows

**Why this matters.** The model was trained on *cleaned* rows (NaN removed,
±Infinity replaced, duplicates dropped). Real SOC data is dirty. This step
samples **raw, untouched CSVs** — including rows the pipeline would have
filtered — runs them through the *same* fit-free feature transform, and
predicts. Lower performance here than on the processed val split is expected
and *informative*: it quantifies the model's robustness to dirty input.

It also uses a **different day's raw file than training data** where possible,
as a crude temporal/generalization check.


In [ ]:
import sys, json, glob
import numpy as np
import pandas as pd
import joblib
from pathlib import Path
sys.path.insert(0, "src")

from sentinelxai.config import get_config
from sentinelxai.features.engineering import apply_feature_engineering

cfg = get_config()

# --- pick raw files: skip Monday (pure BENIGN), sample from attack days ---
raw_files = sorted(glob.glob(str(cfg.paths.data_raw_dir / "*.csv")))
print("Raw files found:", [Path(f).name for f in raw_files])

# Sample N rows from the FIRST attack-bearing file (e.g. Tuesday-WorkingHours).
N_SAMPLE = 50000
df_raw = pd.read_csv(raw_files[1], nrows=N_SAMPLE, low_memory=False)
print("Raw rows sampled:", len(df_raw), "| columns:", len(df_raw.columns))

label_col = cfg.data.label_column
X_raw, _report = apply_feature_engineering(df_raw, cfg.features)
with open("reports/lightgbm/lightgbm_feature_list.json") as fh:
    feats = json.load(fh)["feature_columns"]
X_raw = X_raw.reindex(columns=feats)   # align to the model's exact column order

y_raw = df_raw[label_col].reindex(X_raw.index)  # keep rows the transform kept
print("Test rows after transform:", len(X_raw), "| labels kept:", y_raw.notna().sum())

# --- predict with the trained model + encoder ---
model = joblib.load("models/lightgbm/lightgbm.joblib")
encoder = joblib.load("models/lightgbm/label_encoder.joblib")
proba = model.predict_proba(X_raw.astype(np.float32))
pred_idx = np.argmax(proba, axis=1)
pred = np.array(encoder.classes_)[pred_idx]
conf = proba[np.arange(len(proba)), pred_idx]

# --- compare to ground truth ---
valid = y_raw.notna().to_numpy()
y_true = y_raw.to_numpy()[valid]
y_pred = pred[valid]
acc = float((y_true == y_pred).mean())
print("
===== RAW REAL-LIFE TEST (uncleaned rows) =====")
print(f"Rows: {len(valid)}  Accuracy: {acc:.4f}")
print(pd.crosstab(pd.Series(y_pred, name="pred"), pd.Series(y_true, name="true")).head(20))

out = pd.DataFrame({"true": y_true, "pred": y_pred, "confidence": conf[valid]})
out["correct"] = out["true"] == out["pred"]
out.to_csv("raw_real_life_test.csv", index=False)
print("Saved raw_real_life_test.csv")
print("Misclassification rate on RAW dirty data: {:.4f}".format(1 - acc))


## 8 · Export artifacts for local deployment

Download `sentinelxai_artifacts.zip`, then on your machine:

```
Expand-Archive sentinelxai_artifacts.zip -DestinationPath .
# -> models/lightgbm/lightgbm.joblib + label_encoder.joblib
# -> reports/lightgbm/*  (metrics, feature list)
```

After that, the local FastAPI + Streamlit (`uvicorn ...:8010`, `streamlit run ...`)
will load the **real model** and serve live, SHAP-backed predictions.


In [ ]:
import shutil, os
!mkdir -p artifacts
!cp -r models/lightgbm artifacts/models_lightgbm
!cp -r reports/lightgbm artifacts/reports_lightgbm
!cp raw_real_life_test.csv artifacts/ 2>/dev/null
!cp data/processed/feature_engineering_report.json artifacts/ 2>/dev/null
shutil.make_archive("/content/sentinelxai_artifacts", "zip", "/content/artifacts")
print("Created /content/sentinelxai_artifacts.zip")
from google.colab import files
files.download("/content/sentinelxai_artifacts.zip")
